# Unsupervised Learning for Operations Research

Discovering hidden structures in data to support clustering, segmentation, and facility location problems.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

sns.set(style="whitegrid")
np.random.seed(42)

## Customer Segmentation

In [ ]:
# Generate synthetic customer data for segmentation (relevant to pricing, inventory, service levels)
n_customers = 1200

customer_data = pd.DataFrame({
    'annual_spend': np.random.lognormal(8, 1.2, n_customers).astype(int),
    'order_frequency': np.random.poisson(12, n_customers),
    'avg_order_value': np.random.uniform(40, 450, n_customers),
    'return_rate': np.random.beta(2, 10, n_customers),
    'distance_to_warehouse': np.random.uniform(5, 800, n_customers),
    'loyalty_score': np.random.uniform(0, 100, n_customers)
})

print("Customer dataset preview:")
display(customer_data.head())

In [ ]:
# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(customer_data)

# Determine optimal number of clusters using silhouette score
silhouette_scores = []
k_range = range(2, 10)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)

plt.plot(k_range, silhouette_scores, marker='o')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Choosing Optimal Number of Clusters')
plt.show()

# Fit final model with k=4
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
customer_data['cluster'] = kmeans.fit_predict(X_scaled)

print(customer_data['cluster'].value_counts())

In [ ]:
# Visualize clusters with PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=customer_data['cluster'], palette='viridis', s=60)
plt.title('Customer Segments (PCA Projection)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend(title='Cluster')
plt.show()

## Cluster Analysis and Interpretation

In [ ]:
# Profile each cluster
profile = customer_data.groupby('cluster').mean().round(2)
print("Cluster Profiles:")
display(profile)

## Facility Location Insight (Simple Example)

Using cluster centroids as candidate locations for warehouses or distribution centers.

In [ ]:
# Extract centroids in original scale (approximate)
centroids = scaler.inverse_transform(kmeans.cluster_centers_)
centroid_df = pd.DataFrame(centroids, columns=customer_data.columns[:-1])
centroid_df.index.name = 'Cluster'

print("Approximate cluster centroids (potential facility locations):")
display(centroid_df[['distance_to_warehouse', 'annual_spend']])

## Exercises

- Apply DBSCAN or Hierarchical Clustering and compare results.
- Use the clusters to define differentiated service levels or pricing strategies.
- Integrate cluster assignments into a mixed-integer facility location optimization model.

The next chapter will cover **Reinforcement Learning** applications in Operations Research.